Out of all the figures in the final project, the heat map underwent the largest changes to move to PowerBI. 

In the final version, you get a version of the heat map that is pretty standard - colored blocks, scale on the right w/ numbers, etc. This is due to (no matter how many times I tried) I could not get PowerBI to pull everything over. I believe that this is still user friendly, it is just putting more heavy lifting on the user to hover over the blocks to see the hover text.

I did some consultatory work with Google Gemini - even broke out the Pro version. It wouldn't work no matter how many angles we approached it from. For record keeping, I am going to include a few versions of the heat map here w/ the code. Each heat map references a different frame, hence three slightly different figures. I'll include a comment at the start of each explaining what went wrong.

In [1]:
#Section 0a - Importing Libraries
import pandas as pd #allows for data manipulation and management of dataframes
import numpy as np #allows for additional manipulation logic of data (reading lists, if statements, etc.)
import plotly.graph_objects as go #allows for the creation of figures and visualizations
import plotly.io as pio #allows for conversion of plotly code to HTML file for uploading to PowerBI
import json #allows for figures to be converted to JSON file for entry into HTML code

In [2]:
#Section 0b - Universal Variables

#The Current Term Map
currentterm = [202610] #IMPORTANT - UPDATE FOR EACH NEW CYCLE
termmap = {10: "Spring", 80: "Fall"} #Reads the last two digits to determine term used.
year = str(currentterm[0])[:4] #reads the first 4 digits to determine the term
termsuffix = int(str(currentterm[0])[4:]) #reads the last two digits to determine the semester
prettyterm = f"{termmap.get(termsuffix, 'Semester')} {year}" #This uses the above to spit out the term that is being examined in a readable format

#The Historical Term Map
figtermmap = { #Built up to Spring 2030 - feel free to add/remove terms as appropriate
    202280 : "Fall 2022",
    202310 : "Spring 2023",
    202380 : "Fall 2023",
    202410 : "Spring 2024",
    202480 : "Fall 2024",
    202510 : "Spring 2025",
    202580 : "Fall 2025",
    202610 : "Spring 2026",
    202680 : "Fall 2026",
    202710 : "Spring 2027",
    202780 : "Fall 2027",
    202810 : "Spring 2028",
    202880 : "Fall 2028",
    202910 : "Spring 2029",
    202980 : "Fall 2029",
    203010 : "Spring 2030"
}

#Heat Term Map List
#IMPORTANT - UPDATE TO MOST RECENT TERM THAT DOES NOT HAVE FINAL GRADES
currentheatterm = [202610]

In [3]:
#Section 0g - Heat Map Variables
#Contains variables that help build the heat maps

#Creating the color map that will be used for the heat map
civicustom = [
    [0.0,"#013271"],
    [0.1,"#5E636E"],
    [0.25,"#9D9576"],
    [1.0,"#E7D150"]
]

#Creating a Grade Scale Variable
gradescalenum = [0.0, 1.0, 1.3, 1.7, 2.0, 2.3, 2.7, 3.0, 3.3, 3.7, 4.0]
gradescalestring = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"]

In [4]:
#Section 7a - Reloading the Data
mtheat = pd.read_csv("mthubonlyclean.csv")
mtheat.shape

(58787, 18)

In [5]:
#Section 7b - Removing most recent term w/o Final Grades
mtheat = mtheat.loc[~mtheat["Academic Period"].isin(currentheatterm)]
mtheat.shape

(52328, 18)

In [6]:
mtheat = mtheat.sort_values("Academic Period", ascending = False)
mtheat["Academic Period"].unique()

array([202580, 202510, 202480, 202410, 202380, 202310, 202280])

In [7]:
#Section 7b - Mapping the Terms
mtheat["Academic Period"] = mtheat["Academic Period"].map(figtermmap)
mtheat["Academic Period"].unique()

<StringArray>
[  'Fall 2025', 'Spring 2025',   'Fall 2024', 'Spring 2024',   'Fall 2023',
 'Spring 2023',   'Fall 2022']
Length: 7, dtype: str

In [8]:
#Section 7c - Creating the Term List
heattermlist = mtheat["Academic Period"].unique()

In [9]:
#Section 7c - Creating Unit-Specific Dataframes
aedheat = mtheat[mtheat["Subject"] == "AED"]
archheat = mtheat[mtheat["Subject"] == "ARCH"]
arcsheat = mtheat[mtheat["Subject"] == "ARCS"]
cmgtheat = mtheat[mtheat["Subject"] == "CMGT"]
idheat = mtheat[mtheat["Subject"] == "ID"]
cciheat = mtheat[mtheat["Department"] == "CCI"]
commheat = mtheat[mtheat["Department"] == "COMM"]
ematheat = mtheat[mtheat["Department"] == "EMAT"]
mdjheat = mtheat[mtheat["Department"] == "MDJ"]
vcdheat = mtheat[mtheat["Department"] == "VCD"]
artheat = mtheat[mtheat["Department"] == "ART"]
fdmheat = mtheat[mtheat["Department"] == "FDM"]
musheat = mtheat[mtheat["Department"] == "MUS"]
thdnheat = mtheat[mtheat["Department"] == "THDN"]

In [10]:
#The Original Code
# This is the "base" version of my heat map. It is frankly the figure I'm the most proud of, but it wouldn't carry over the numbers to PowerBI when I tried moving it via the HTML Visualizer.
buttons = []
zmatrix = {}
textmatrix = {}

initial_term = heattermlist[0]

for term in heattermlist:
    termframe = aedheat[aedheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    textmatrix[term] = termcrosstab.astype(str).values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ #these are what update every term
            "z": [zmatrix[term]], 
            "text": [textmatrix[term]],
            "zmin": [zbot],
            "zmax": [ztop],
            "colorbar.tickvals": [[ztop*.05, ztop/2, ztop*.95]]
        }]
    ))



recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    text = textmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{text}",
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [ztop*.05, ztop/2, ztop*.95],
        ticktext = ["Lower", "Mid", "Higher"],
        title = dict( text = "<b>Student Density</b>",
        font_color = "black")
        ),
    visible = True)

pastheattrace = go.Heatmap(
    z = zmatrix[term],
    text = textmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{text}",
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [ztop*.05, ztop/2, ztop*.95],
        ticktext = ["Lower", "Mid", "Higher"],
        title = "<b>Student Density</b>"
        ),
    visible = False)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "AED Grade Movement from Midterms to Final",
        font_weight = "bold", 
        font_size = 24,
        font_color = "black",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .97),
    plot_bgcolor = "black",
    xaxis_showgrid = False,
    yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 600,
    width = 700,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .6,
        y = 1.105
    )],
    legend = dict(
        orientation = "h",
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = -.1),
    font_family = "Segoe UI"
    )

aedheat = go.Figure(data = [recentheattrace, pastheattrace, legendtrace], layout = layout)

aedheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3125,
    y = 1.098,
    xref = "paper",
    yref = "paper",
    font_color = "black",
    font_size = 16
)
aedheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

aedheat.show()

In [12]:
#Gemini's First Attempt - Adding lots of numbers
#Gemini wanted to try adding annotations that would change when you changed the term. Each annotation would be a different number for the block. 
#However, it couldn't get the formatting right, leading to the numbers all crumpling in the corner. I think this code has some merit and *could* be salvagable with more time
import pandas as pd
import plotly.graph_objects as go

#Section 7d - AED Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

# 1. We must define the "Semester:" label here so it doesn't get erased by the dropdown!
base_annotations = [
    dict(
        text = "<b>Semester:</b>",
        showarrow = False,
        x = .3125, y = 1.098,
        xref = "paper", yref = "paper",
        font = dict(color = "black", size = 16)
    )
]

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])

    # 2. Build the text grid manually for this specific term
    term_annotations = list(base_annotations) # Start with the Semester label
    
    for i, row in enumerate(zmatrix[term]):      # i is the row index (Final Grade / y-axis)
        for j, val in enumerate(row):            # j is the col index (Mid Term / x-axis)
            
            # Dynamic Contrast: White text on dark cells, black text on light cells!
            text_color = "white" if val > (ztop / 2) else "black"
            
            term_annotations.append(dict(
                x = gradescalestring[j], 
                y = gradescalestring[i], 
                text = str(val),
                showarrow = False,
                font = dict(color = text_color, size = 12)
            ))

    # 3. Pass BOTH the Data updates and the Layout Annotation updates to the button
    buttons.append(dict(
        method = "update",
        label = term,
        args = [
            { # Trace Updates (Data)
                "z": [zmatrix[term]], 
                "zmin": [zbot],
                "zmax": [ztop],
                "colorbar.tickvals": [[ztop*.05, ztop/2, ztop*.95]]
            },
            { # Layout Updates: Push the new numbers to the grid!
                "annotations": term_annotations
            }
        ]
    ))

# 4. We ONLY need one heatmap trace now!
recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    # DELETED text and texttemplate parameters here!
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [ztop*.05, ztop/2, ztop*.95],
        ticktext = ["Lower", "Mid", "Higher"],
        title = dict(text = "<b>Student Density</b>", font_color = "black")
    ),
    visible = True
)

legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(color = "#FF8C00", width = 2, dash = "dash"),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "AED Grade Movement from Midterms to Final",
        font_weight = "bold", font_size = 24, font_color = "black",
        xanchor = "center", x = .5, yanchor = "top", y = .97),
    plot_bgcolor = "black",
    xaxis_showgrid = False, yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 600, width = 700,
    updatemenus = [dict(
        buttons = buttons,
        direction = "down", showactive = True,
        xanchor = "center", yanchor = "top", x = .6, y = 1.105
    )],
    legend = dict(orientation = "h", xanchor = "center", x = .5, yanchor = "top", y = -.1),
    font_family = "Segoe UI",
    
    # 5. Set the initial numbers on load using the first term's annotations
    annotations = buttons[0]["args"][1]["annotations"]
)

# 6. Build the figure with just the ONE heatmap and the legend
aedheat = go.Figure(data = [recentheattrace, legendtrace], layout = layout)

# The shape stays completely separate from annotations, so it won't be overwritten
aedheat.add_shape(
    type = "rect",
    line = dict(color = "#FF8C00", width = 2, dash = "dash"),
    x0 = -0.5, x1 = 4.5, y0 = -0.5, y1 = 10.5, layer = "above"
)

aedheat.show()

In [13]:
#Bridging the Gap w/ Scatter Plot
#I proposed combining the method proposed w/ annotations with a scatter plot to see if it worked. It did (kind of). The numbers overlaid, with hte main issue being the color of the numbers.
#This wasn't ideal, but figure it could be workshopped after this pass. It had the same problem of not carrying over to PowerBI like the first example though.
import pandas as pd
import plotly.graph_objects as go

#Section 7d - AED Heat Map
buttons = []
zmatrix = {}

initial_term = heattermlist[0]

# 1. Create flat coordinate lists for the Scatter trace X and Y
flat_x = []
flat_y = []
for i, y_val in enumerate(gradescalestring): # Rows (Final Grades)
    for j, x_val in enumerate(gradescalestring): # Cols (Mid Term Grades)
        flat_x.append(x_val)
        flat_y.append(y_val)

# Dictionaries to store the flat text and colors for each term
text_dict = {}
color_dict = {}

for term in heattermlist:
    termframe = artheat[artheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])
    termcrosstab = termcrosstab.reindex(index = gradescalenum, columns = gradescalenum, fill_value = 0)

    zmatrix[term] = termcrosstab.values.tolist()

    ztop = max(max(row) for row in zmatrix[term]) 
    zbot = min(min(row) for row in zmatrix[term])
    
    # 2. Build the flat text and color lists for this specific term
    term_text = []
    term_color = []
    for i, row in enumerate(zmatrix[term]):      
        for j, val in enumerate(row):            
            term_text.append(str(val))
            # Keep our dynamic contrast!
            term_color.append("white" if val > (ztop / 2) else "black")
            
    text_dict[term] = term_text
    color_dict[term] = term_color

    # 3. Update both the Heatmap (Trace 0) and Scatter (Trace 1), ignore Legend (Trace 2)
    # The arrays in args correspond to [Trace 0, Trace 1, Trace 2]
    buttons.append(dict(
        method = "update",
        label = term,
        args = [{ 
            "z": [zmatrix[term], None, None], 
            "zmin": [zbot, None, None],
            "zmax": [ztop, None, None],
            "colorbar.tickvals": [[ztop*.05, ztop/2, ztop*.95], None, None],
            "text": [None, text_dict[term], None],
            "textfont.color": [None, color_dict[term], None]
        }]
    ))

# Trace 0: The Heatmap
recentheattrace = go.Heatmap(
    z = zmatrix[initial_term],
    zmin = zbot,
    zmax = ztop,
    x = gradescalestring, 
    y = gradescalestring,
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [ztop*.05, ztop/2, ztop*.95],
        ticktext = ["Lower", "Mid", "Higher"],
        title = dict(text = "<b>Student Density</b>", font_color = "black")
    ),
    visible = True
)

# Trace 1: The Scatter Text Overlay
textoverlaytrace = go.Scatter(
    x = flat_x,
    y = flat_y,
    text = text_dict[initial_term],
    mode = "text",
    textfont = dict(color = color_dict[initial_term], size = 12),
    hoverinfo = "skip", # Critical so it doesn't block the heatmap hover tooltips!
    showlegend = False
)

# Trace 2: The Legend
legendtrace = go.Scatter(
    x = [None],
    y = [None],
    mode = "lines",
    line = dict(color = "#FF8C00", width = 2, dash = "dash"),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "AED Grade Movement from Midterms to Final",
        font_weight = "bold", font_size = 24, font_color = "black",
        xanchor = "center", x = .5, yanchor = "top", y = .97),
    plot_bgcolor = "black",
    xaxis_showgrid = False, yaxis_showgrid = False,
    xaxis = dict(title = dict(text = "<i>Mid Term Grades</i>", font_size = 18)),
    yaxis = dict(title = dict(text = "<i>Final Grades</i>", font_size = 18)),
    height = 600, width = 700,
    updatemenus = [dict(
        buttons = buttons,
        direction = "down", showactive = True,
        xanchor = "center", yanchor = "top", x = .6, y = 1.105
    )],
    legend = dict(orientation = "h", xanchor = "center", x = .5, yanchor = "top", y = -.1),
    font_family = "Segoe UI"
)

# Add all THREE traces to the figure
aedheat = go.Figure(data = [recentheattrace, textoverlaytrace, legendtrace], layout = layout)

aedheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3125, y = 1.098,
    xref = "paper", yref = "paper",
    font_color = "black", font_size = 16
)

aedheat.add_shape(
    type = "rect",
    line = dict(color = "#FF8C00", width = 2, dash = "dash"),
    x0 = -0.5, x1 = 4.5, y0 = -0.5, y1 = 10.5, layer = "above"
)

aedheat.show()